# Create Your Own Higher-Order Functions

*Codecademy MLE Path — Functional programming.*

Build two higher-order functions out of `filter()`, `map()`, `reduce()` and recursion, then use them on warehouse sales data.

| function | args | returns | built from |
|---|---|---|---|
| `count` | `predicate, itr` | how many elements of `itr` make `predicate` true | `filter()` + `reduce()` |
| `average` | `itr` | arithmetic mean of `itr` | recursive helper + `next()` |

> **Data:** `1kSalesRec.csv` is the real Codecademy file (1000 orders, 14 columns). Note that a few country names carry a
> trailing space (`"Mauritius "`, `"Moldova "`, `"Tunisia "`…), so an exact `==` match on those would miss. Belgium and Portugal are clean.

In [ ]:
import csv
from functools import reduce

---
## Part 1 — `count()`

*(Codecademy's starter names the parameter `iterable`, while the instructions say `itr`. Same thing.)*

**Step 1.** `filter()` `iterable` by `predicate` → save as `count_filter`.

**Step 2.** `reduce()` over `count_filter` so the accumulator goes up by **1 per surviving element** (not by the element's value).
Save as `count_reduce` and return it.

Remember the third argument of `reduce()` sets the **initial value**, and it doesn't have to be the same type as the elements.

In [ ]:
def count(predicate, iterable):
  pass # Remove this keyword when you start working on this function.

In [ ]:
print(count(lambda x: x == 3, [1, 3, 3, 5, 3]))   # expected: 3
print(count(lambda s: s.startswith("a"), ["apple", "avocado", "banana"]))   # expected: 2

### Check — Part 1

In [ ]:
def check_part1():
    assert count(lambda x: x == 3, [1, 3, 3, 5, 3]) == 3, "should count the three 3s"
    assert count(lambda x: x == 99, [1, 2, 3]) == 0, "no matches should return 0, not None or an error"
    assert count(lambda x: x > 0, [5, 5, 5]) == 3, "should count elements, not add them up (5+5+5=15)"
    assert count(lambda s: s == "b", iter(["a", "b", "b"])) == 2, "should work on an iterator too"
    print("Part 1 OK")
check_part1()

---
## Part 2 — `average()` (recursive)

The loop you're rewriting recursively:
```python
curr_count = 0
curr_sum = 0
for i in iterable:
    curr_sum += i
    curr_count += 1
```

**Step 3.** Inside `average()`, define `avg_helper(curr_count, itr, curr_sum)`.

**Step 4.** `next_num = next(itr, "null")`. Add it to `curr_sum`, increment `curr_count`.

**Step 5.** If `next_num` is `"null"` → you're out of elements → return the average.

**Step 6.** Otherwise recurse with the updated `curr_count` and `curr_sum`, and **return** that call's result.

**Step 7.** Order inside the helper: **base case → computation → recursive call.**

**Step 8.** In `average()`, make the first call to `avg_helper` with the right starting values and return it.

The starter already does `iterable = iter(itr)`, so `average()` accepts a plain list too, but only if `avg_helper`
gets that `iterable` and not the original `itr`. `next()` needs an **iterator**, and `next([1, 2])` raises a `TypeError`.

In [ ]:
def average(itr):
  # If itr is not iterable, this will generate an iterator.
  iterable = iter(itr)

In [ ]:
print(average(iter([2, 4, 6])))          # expected: 4.0
print(average(map(float, ["1.5", "2.5"])))   # expected: 2.0

### Check — Part 2

In [ ]:
def check_part2():
    assert average(iter([2, 4, 6])) == 4, "average of 2, 4, 6 is 4"
    assert average(iter([10])) == 10, "one element: average is that element (check your count isn't off by one)"
    assert abs(average(iter([1, 2, 3, 4])) - 2.5) < 1e-9, "average of 1..4 is 2.5"
    assert abs(average(map(float, ["1.5", "2.5"])) - 2.0) < 1e-9, "should work on a map object"
    print("Part 2 OK")
check_part2()

---
## Part 3 — Using them on `1kSalesRec.csv`

Fields (index in brackets): Region[0], **Country[1]**, Item Type[2], Sales Channel[3], Order Priority[4], Order Date[5],
Order ID[6], Ship Date[7], Units Sold[8], Unit Price[9], Unit Cost[10], Total Revenue[11], Total Cost[12], **Total Profit[13]**

Every record comes in as a **list of strings**.

**Step 9.** Replace `count_belgiums = None` with a `count()` call using `lambda x: x[1] == "Belgium"`. What's the second argument?

**Step 10.** Replace `avg_portugal = None` with an `average()` call on Portugal's Total Profit: `filter()` to Portugal rows, `map()` to
`float(row[13])`. Think about which one has to happen first.

`csvfile.seek(0)` rewinds the file, because the reader is used up after Step 9.

In [ ]:
with open('1kSalesRec.csv', newline = '') as csvfile:
  reader = csv.reader(csvfile, delimiter=',', quotechar='|')
  fields = next(reader)
  count_belgiums = None
  print(count_belgiums)
  csvfile.seek(0)
  avg_portugal = None
  print(avg_portugal)

### Check — Part 3

In [ ]:
def check_part3():
    # reference answers with plain loops, independent of your functions
    with open('1kSalesRec.csv', newline='') as f:
        rows = list(csv.reader(f))[1:]
    ref_count = sum(1 for r in rows if r[1] == "Belgium")
    port = [float(r[13]) for r in rows if r[1] == "Portugal"]
    ref_avg = sum(port) / len(port)
    assert count_belgiums is not None, "Step 9: count_belgiums is still None"
    assert count_belgiums == ref_count, f"Step 9: expected {ref_count} Belgium orders, got {count_belgiums}"
    assert avg_portugal is not None, "Step 10: avg_portugal is still None"
    assert abs(avg_portugal - ref_avg) < 1e-6, f"Step 10: expected {ref_avg:.2f}, got {avg_portugal}"
    print(f"Part 3 OK — Belgium orders: {count_belgiums}, Portugal avg profit: {avg_portugal:,.2f}")
check_part3()

---
## Step 11 — Going further (optional)

Ideas: other countries/columns, a recursive `median` or `standard_deviation`, other ways to get the Step 9/10 answers.

⚠️ Python's recursion limit is about 1000 calls. `average()` makes one call per element, so averaging **all** ~1000 rows
can raise `RecursionError`. Portugal alone is fine.

In [ ]:
# your extensions here